<figure>
    <picture>
        <source srcset="assets/iita-logo.png" style="max-height: 10rem;">
        <img src="https://raw.githubusercontent.com/Computational-Biology-Aachen/2026-photosynthesis-hackathon-template/refs/heads/main/assets/iita-logo.png" style="max-height: 10rem;">
    </picture>
</figure>

# Cowpea data

This notebook contains example analysis for the cowpea data kindly supplied by the IITA

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import ticker

from src import (
    drop_na_multiple,
    heritability,
    heritability_with_covariates,
    load_iita_cowpea,
    plot,
)

desc, s1, s2 = load_iita_cowpea(Path("data"))
desc.head()

## EDA

In [ ]:
# Basic checks for completeness

fig, axs = plot.fig_axs(nrows=1, ncols=3, figsize=(12, 3), sharex=False)
fig.suptitle("Counts by hyper-parameter")

for ax, (par, title) in zip(
    axs,
    [
        ("GEN", "Genotype"),
        ("ENV", "Environment (location x year x  stress condition)"),
        ("BLK", "Block within replications"),
    ],
    strict=False,
):
    (
        s1[par]
        .value_counts()
        .sort_index()
        .plot(kind="bar", title=title, ylabel="Count", ax=ax)
    )
    ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
plt.show()

In [ ]:
# Overall distributions

fig, axs = plot.fig_axs(nrows=1, ncols=7, figsize=(12, 4), sharex=False)
fig.suptitle("Overall parameter distributions")

for ax, (par, title) in zip(
    axs,
    [
        ("RCC", "Relative chlorophyll content"),
        ("LTD", "Leaf temperature differential"),
        ("LEF", "Linear electron flow"),
        ("NPQt", "Non-photochemical quenching"),
        ("ΦII", "Quantum yield of photosystem II"),
        ("ΦNO", "Ratio of incoming light lost"),
        ("ΦNPQ", "Ratio of incoming light towards NPQ"),
    ],
    strict=False,
):
    sns.violinplot(s1.loc[:, [par]], ax=ax)
    ax.set_ylabel(title)
plt.show()

In [ ]:
fig, axs = plot.fig_axs(nrows=2, ncols=2, figsize=(8, 6))
fig.suptitle("Parameters by environment")

for ax, (par, title) in zip(
    axs,
    [
        ("RCC", "Relative chlorophyll content"),
        ("LTD", "Leaf temperature differential"),
        ("LEF", "Linear electron flow"),
        ("NPQt", "Non-photochemical quenching"),
        ("ΦII", "Quantum yield of photosystem II"),
        ("ΦNO", "Ratio of incoming light lost"),
        ("ΦNPQ", "Ratio of incoming light towards NPQ"),
    ],
    strict=False,
):
    sns.violinplot(
        data=s1,
        x="ENV",
        y=par,
        ax=ax,
    )
    ax.set_title(title)
plt.show()

In [ ]:
fig, axs = plot.fig_axs(nrows=2, ncols=2, figsize=(8, 6))
fig.suptitle("Parameters by genotype")

for ax, (par, title) in zip(
    axs,
    [
        ("RCC", "Relative chlorophyll content"),
        ("LTD", "Leaf temperature differential"),
        ("LEF", "Linear electron flow"),
        ("NPQt", "Non-photochemical quenching"),
        ("ΦII", "Quantum yield of photosystem II"),
        ("ΦNO", "Ratio of incoming light lost"),
        ("ΦNPQ", "Ratio of incoming light towards NPQ"),
    ],
    strict=False,
):
    sns.violinplot(
        data=s1,
        x="GEN",
        y=par,
        ax=ax,
    )
    ax.set_title(title)
plt.show()

## Heritability

In [ ]:
# Clean up data before analysis
data, env_factors, gtype = drop_na_multiple(
    s1.loc[:, ["LEF", "NPQt", "ΦII", "ΦNO", "ΦNPQ"]],
    s1.loc[:, ["ENV"]],
    s1.loc[:, "GEN"],
)

# Filter out genotypes with fewer than 3 entries
more_than_3 = gtype.groupby(gtype).transform("size") > 3
data = data[more_than_3]
env_factors = env_factors[more_than_3]

In [ ]:
h2 = heritability(
    data=data,
    gtype=gtype,
)

_ = h2.plot()

In [ ]:
h2_cov = heritability_with_covariates(
    data=data,
    env_factors=env_factors,
    gtype=gtype,
)

_ = h2_cov.plot()